# Exploratory Data Analysis (EDA) for Fire Detection Dataset

This notebook explores the fire detection dataset, visualizes samples, and analyzes class distribution.

In [ ]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

# Add parent directory to path
sys.path.append('..')

from src.datasets import FireDetectionDataset

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Dataset Overview

In [ ]:
# Define paths
data_root = '../data'
train_img_dir = os.path.join(data_root, 'train', 'images')
train_label_dir = os.path.join(data_root, 'train', 'labels')

# Load dataset
dataset = FireDetectionDataset(
    img_dir=train_img_dir,
    label_dir=train_label_dir,
    img_size=640,
    augment=False
)

print(f"Total samples: {len(dataset)}")
print(f"Class names: {dataset.class_names}")

## 2. Class Distribution

In [ ]:
# Get class distribution
class_dist = dataset.get_class_distribution()

# Plot
plt.figure(figsize=(10, 6))
plt.bar(class_dist.keys(), class_dist.values(), color=['orange', 'gray'])
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class Distribution in Training Set')
plt.grid(True, alpha=0.3)

for i, (k, v) in enumerate(class_dist.items()):
    plt.text(i, v + 50, str(v), ha='center', fontsize=12)

plt.show()

print(f"\nClass Distribution:")
for name, count in class_dist.items():
    print(f"  {name}: {count} ({count/sum(class_dist.values())*100:.1f}%)")

## 3. Sample Visualization

In [ ]:
def visualize_sample(idx):
    """Visualize a single sample with bounding boxes."""
    # Load image
    img_path = dataset.img_files[idx]
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    
    # Load labels
    label_path = dataset.label_files[idx]
    boxes, labels = dataset._load_yolo_labels(label_path)
    
    # Convert YOLO to xyxy format
    boxes_xyxy = dataset.yolo_to_xyxy(boxes, w, h)
    
    # Draw boxes
    for box, label in zip(boxes_xyxy, labels):
        x1, y1, x2, y2 = box.astype(int)
        color = (255, 165, 0) if label == 0 else (128, 128, 128)  # Orange for flame, gray for smoke
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        
        # Label text
        text = dataset.class_names[label]
        cv2.putText(image, text, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    return image

# Visualize random samples
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i in range(6):
    idx = np.random.randint(0, len(dataset))
    img = visualize_sample(idx)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'Sample {idx}')

plt.tight_layout()
plt.show()

## 4. Bounding Box Statistics

In [ ]:
# Collect box statistics
box_widths = []
box_heights = []
box_areas = []
boxes_per_image = []

for i in range(len(dataset)):
    label_path = dataset.label_files[i]
    boxes, _ = dataset._load_yolo_labels(label_path)
    
    boxes_per_image.append(len(boxes))
    
    for box in boxes:
        _, _, w, h = box
        box_widths.append(w)
        box_heights.append(h)
        box_areas.append(w * h)

# Plot distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(box_widths, bins=50, alpha=0.7, color='blue')
axes[0, 0].set_xlabel('Box Width (normalized)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Box Widths')

axes[0, 1].hist(box_heights, bins=50, alpha=0.7, color='green')
axes[0, 1].set_xlabel('Box Height (normalized)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Box Heights')

axes[1, 0].hist(box_areas, bins=50, alpha=0.7, color='red')
axes[1, 0].set_xlabel('Box Area (normalized)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Box Areas')

axes[1, 1].hist(boxes_per_image, bins=range(max(boxes_per_image)+2), alpha=0.7, color='purple')
axes[1, 1].set_xlabel('Number of Boxes')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Boxes per Image')

plt.tight_layout()
plt.show()

print(f"\nBox Statistics:")
print(f"  Avg width: {np.mean(box_widths):.3f}")
print(f"  Avg height: {np.mean(box_heights):.3f}")
print(f"  Avg area: {np.mean(box_areas):.3f}")
print(f"  Avg boxes per image: {np.mean(boxes_per_image):.1f}")

## 5. Summary and Insights

Based on the EDA:

1. **Class Distribution**: Check if classes are balanced or if data augmentation/loss weighting is needed
2. **Object Scales**: Understand the typical size range for fire/smoke detection
3. **Image Quality**: Ensure images are clear and properly labeled
4. **Data Diversity**: Look for variety in lighting, backgrounds, and scenarios

This analysis helps inform:
- Choice of augmentation strategies
- Loss function selection
- Model architecture decisions
- Training hyperparameters